In [2]:
import sys
import os
project_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_path)

In [6]:
from gensim.models import Word2Vec
from nlp_utils import sentence_filter
import pandas as pd
import spacy
import numpy as np
from gensim.test.utils import common_texts

In [23]:
book = pd.read_csv("../../resources/historia_da_arte.csv")

In [24]:
book_sub = book[5:33]
model = spacy.load('pt_core_news_sm')
content = " ".join(book_sub["Chapter Content"].to_list()).lower()

In [91]:
doc = model(content)

In [92]:
sentences = sentence_filter(doc=doc, excluded_words=["Fig"])


In [93]:
# Assuming 'sentences' is a list of tokenized sentences
model_wv = Word2Vec(sentences=sentences, vector_size=100000, window=5, min_count=1, workers=12)

In [ ]:
sims = model_wv.wv.most_similar('leonardo')
sims

In [ ]:
for document_number, score in sorted(sims, key=lambda x: x[1], reverse=True):
    print(document_number, score)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Extract word vectors
words = list(model_wv.wv.key_to_index)
vectors = [model_wv.wv[word] for word in words]

# Reduce to 2 dimensions with PCA
pca = PCA(n_components=10)
result = pca.fit_transform(vectors)

# Plot
plt.scatter(result[:, 0], result[:, 1])
for i, word in enumerate(words):
    plt.annotate(word, xy=(result[i, 0], result[i, 1]))
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sample dataset
data = pd.DataFrame({
    'first': ['van', 'da', 'Michelangelo'],
    'second': ['Eyck', 'Vinci', 'Buonarroti'],
    'total_importance': [1000000, 0, 2000000],
    'paragraph': [3, 2, 4],
    'sentence': [9, 7, 12]
})

# Combine first and second columns
data['combined'] = data['first'] + ' ' + data['second']

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(data['combined'])

# Calculate cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix)

# Normalize total_importance
max_importance = data['total_importance'].max()
normalized_importance = data['total_importance'] / max_importance

# Calculate weighted similarity
def weighted_similarity(idx1, idx2):
    text_sim = cosine_sim[idx1, idx2]
    importance_sim = 1 - abs(normalized_importance[idx1] - normalized_importance[idx2])
    
    # Give more weight to total_importance
    weighted_sim = text_sim * importance_sim
    return weighted_sim

# Calculate similarity matrix
n = len(data)
similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        similarity_matrix[i, j] = weighted_similarity(i, j)

# Print similarity matrix
print("Similarity Matrix:")
print(similarity_matrix)

# Example: Find most similar pair
max_sim = 0
max_pair = None

for i in range(n):
    for j in range(i+1, n):
        if similarity_matrix[i, j] > max_sim:
            max_sim = similarity_matrix[i, j]
            max_pair = (i, j)

if max_pair:
    print("\nMost similar pair:")
    print(f"1. {data['combined'].iloc[max_pair[0]]}")
    print(f"2. {data['combined'].iloc[max_pair[1]]}")
    print(f"Similarity score: {max_sim:.4f}")

In [31]:
co = pd.read_csv("../../cooccurences.csv")

In [32]:
df = co
df[['x', 'y']] = df['key'].str.split('_', expand=True)

# Create a copy of the DataFrame with flipped x and y
df_flipped = df.copy()
df_flipped['x'], df_flipped['y'] = df_flipped['y'], df_flipped['x']
df_flipped['key'] = df_flipped['x'] + '_' + df_flipped['y']

# Combine original and flipped DataFrames
df_combined = pd.concat([df, df_flipped]).drop_duplicates(subset=['key'])

# Get unique values for x and y (which are now the same)
unique_values = sorted(set(df_combined['x'].unique()) | set(df_combined['y'].unique()))
value_mapping = {val: i for i, val in enumerate(unique_values)}

# Create an empty square matrix
matrix_size = len(unique_values)
matrix = np.zeros((matrix_size, matrix_size))

# Fill the matrix with total_importance values
for _, row in df_combined.iterrows():
    x_idx = value_mapping[row['x']]
    y_idx = value_mapping[row['y']]
    matrix[x_idx, y_idx] = row['total_importance']

# # Print the resulting matrix
# print("Resulting square matrix:")
# print(matrix)

# Create a DataFrame from the matrix for better visualization
matrix_df = pd.DataFrame(matrix, index=unique_values, columns=unique_values)



In [33]:
def get_top_n_y_per_x(matrix_df, n=3, x=None):
    # Create an empty list to store results
    results = []

    # If x is provided, filter the DataFrame to only that x value
    if x is not None:
        if x not in matrix_df.index:
            raise ValueError(f"The provided x value '{x}' is not in the matrix index.")
        rows_to_process = matrix_df.loc[[x]]
    else:
        rows_to_process = matrix_df

    # Iterate through each row (x value)
    for x, row in rows_to_process.iterrows():
        # Sort the row by value in descending order and get top n
        top_n = row.nlargest(n)
        
        # Add each of the top n results to the results list
        for y, value in top_n.items():
            results.append({'x': x, 'y': y, 'value': value})

    # Create a DataFrame from the results
    result_df = pd.DataFrame(results)

    # Sort the result by x and then by value (descending) for better readability
    result_df = result_df.sort_values(['x', 'value'], ascending=[True, False])

    return result_df

# Example usage:

# Get top 3 y values for each x
top_3_all_x = get_top_n_y_per_x(matrix_df, n=3)
print("Top 3 highest values of y for each x:")
print(top_3_all_x)

# Get top 3 y values for a specific x (replace 'specific_x_value' with an actual x value from your data)
specific_x_value = 'leonardo'  # Replace this with an actual x value from your data
try:
    top_3_specific_x = get_top_n_y_per_x(matrix_df, n=3, x=specific_x_value)
    print(f"\nTop 3 highest values of y for x = {specific_x_value}:")
    print(top_3_specific_x)
except ValueError as e:
    print(f"Error: {e}")

# Get overall top 3 values
overall_top_3 = top_3_all_x.nlargest(3, 'value')
print("\nOverall top 3 highest values:")
print(overall_top_3)

Top 3 highest values of y for each x:
            x         y  value
0    adoração      adão    1.0
1    adoração   afresco    1.0
2    adoração     apolo    1.0
3        adão  adoração    1.0
4        adão   afresco    1.0
..        ...       ...    ...
739    árvore  detalhes    3.0
740    árvore      arte    2.0
741      óleo   detalhe    3.0
742      óleo  detalhes    3.0
743      óleo      eyck    3.0

[744 rows x 3 columns]
Error: The provided x value 'leonardo' is not in the matrix index.

Overall top 3 highest values:
            x         y  value
183   detalhe  detalhes   18.0
186  detalhes   detalhe   18.0
639       san     santo   18.0


In [ ]:
list(combinations(["A","B"], 2))